## [Content-based filtering]

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from ast import literal_eval
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# pd.set_option('display.max_rows', None)
# pd.set_option('display.max_columns', None)

In [2]:
movie_data=pd.read_csv("movies_metadata.csv")
# movie_data=movie_data.loc[movie_data["original_language"]=="en", :]
# movie_data=movie_data[["id", "title", "original_language", "genres"]]
movie_data=movie_data.loc[movie_data["original_language"]=="en", ["id", "title", "original_language", "genres"]]

print(movie_data)

           id                        title original_language  \
0         862                    Toy Story                en   
1        8844                      Jumanji                en   
2       15602             Grumpier Old Men                en   
3       31357            Waiting to Exhale                en   
4       11862  Father of the Bride Part II                en   
...       ...                          ...               ...   
45459  222848              Caged Heat 3000                en   
45460   30840                   Robin Hood                en   
45463   67758                     Betrayal                en   
45464  227506             Satan Triumphant                en   
45465  461257                     Queerama                en   

                                                  genres  
0      [{'id': 16, 'name': 'Animation'}, {'id': 35, '...  
1      [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...  
2      [{'id': 10749, 'name': 'Romance'}, {'id': 35, .

C:\Users\jkpxtwin\AppData\Local\Temp\ipykernel_3980\175109430.py:1: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movie_data=pd.read_csv("movies_metadata.csv")


In [3]:
movie_keywords=pd.read_csv("keywords.csv")

print(movie_keywords)

           id                                           keywords
0         862  [{'id': 931, 'name': 'jealousy'}, {'id': 4290,...
1        8844  [{'id': 10090, 'name': 'board game'}, {'id': 1...
2       15602  [{'id': 1495, 'name': 'fishing'}, {'id': 12392...
3       31357  [{'id': 818, 'name': 'based on novel'}, {'id':...
4       11862  [{'id': 1009, 'name': 'baby'}, {'id': 1599, 'n...
...       ...                                                ...
46414  439050             [{'id': 10703, 'name': 'tragic love'}]
46415  111109  [{'id': 2679, 'name': 'artist'}, {'id': 14531,...
46416   67758                                                 []
46417  227506                                                 []
46418  461257                                                 []

[46419 rows x 2 columns]


In [4]:
movie_data.id=movie_data.id.astype(int)
movie_keywords.id=movie_keywords.id.astype(int)
movie_data=pd.merge(movie_data, movie_keywords, on="id")

print(movie_data)

           id                        title original_language  \
0         862                    Toy Story                en   
1        8844                      Jumanji                en   
2       15602             Grumpier Old Men                en   
3       31357            Waiting to Exhale                en   
4       11862  Father of the Bride Part II                en   
...       ...                          ...               ...   
32847  222848              Caged Heat 3000                en   
32848   30840                   Robin Hood                en   
32849   67758                     Betrayal                en   
32850  227506             Satan Triumphant                en   
32851  461257                     Queerama                en   

                                                  genres  \
0      [{'id': 16, 'name': 'Animation'}, {'id': 35, '...   
1      [{'id': 12, 'name': 'Adventure'}, {'id': 14, '...   
2      [{'id': 10749, 'name': 'Romance'}, {'id': 35

In [5]:
# [Study]
movie_data["genres"]=movie_data["genres"].apply(literal_eval)
movie_data["genres"]=movie_data["genres"].apply(lambda x : [d["name"] for d in x]).apply(lambda x : " ".join(x))
movie_data["keywords"]=movie_data["keywords"].apply(literal_eval)
movie_data["keywords"]=movie_data["keywords"].apply(lambda x : [d["name"] for d in x]).apply(lambda x : " ".join(x))

print(movie_data)

           id                        title original_language  \
0         862                    Toy Story                en   
1        8844                      Jumanji                en   
2       15602             Grumpier Old Men                en   
3       31357            Waiting to Exhale                en   
4       11862  Father of the Bride Part II                en   
...       ...                          ...               ...   
32847  222848              Caged Heat 3000                en   
32848   30840                   Robin Hood                en   
32849   67758                     Betrayal                en   
32850  227506             Satan Triumphant                en   
32851  461257                     Queerama                en   

                         genres  \
0       Animation Comedy Family   
1      Adventure Fantasy Family   
2                Romance Comedy   
3          Comedy Drama Romance   
4                        Comedy   
...                  

In [6]:
# [Study]
tfidf_vectorizer=TfidfVectorizer()
tfidf_matrix=tfidf_vectorizer.fit_transform(movie_data["genres"]+" "+movie_data["keywords"]).toarray()
tfidf_matrix_feature=tfidf_vectorizer.get_feature_names_out()
tfidf_matrix=pd.DataFrame(tfidf_matrix, columns=tfidf_matrix_feature, index=movie_data.title)

print(tfidf_matrix)

                             077   10   11   13  1500s  15th  16th  17th  \
title                                                                      
Toy Story                    0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Jumanji                      0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Grumpier Old Men             0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Waiting to Exhale            0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Father of the Bride Part II  0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
...                          ...  ...  ...  ...    ...   ...   ...   ...   
Caged Heat 3000              0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Robin Hood                   0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Betrayal                     0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Satan Triumphant             0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   
Queerama                     0.0  0.0  0.0  0.0    0.0   0.0   0.0   0.0   

           

In [7]:
# [Study]
cos_sim=cosine_similarity(tfidf_matrix)

print(cos_sim)

[[1.         0.04156863 0.00870783 ... 0.         0.         0.        ]
 [0.04156863 1.         0.         ... 0.         0.         0.        ]
 [0.00870783 0.         1.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 1.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


In [8]:
cos_sim_df=pd.DataFrame(cos_sim, index=movie_data.title, columns=movie_data.title)

print(cos_sim_df)

title                        Toy Story   Jumanji  Grumpier Old Men  \
title                                                                
Toy Story                     1.000000  0.041569          0.008708   
Jumanji                       0.041569  1.000000          0.000000   
Grumpier Old Men              0.008708  0.000000          1.000000   
Waiting to Exhale             0.006937  0.065065          0.035846   
Father of the Bride Part II   0.005595  0.000000          0.010906   
...                                ...       ...               ...   
Caged Heat 3000               0.000000  0.000000          0.000000   
Robin Hood                    0.000000  0.000000          0.106819   
Betrayal                      0.000000  0.000000          0.000000   
Satan Triumphant              0.000000  0.000000          0.000000   
Queerama                      0.000000  0.000000          0.000000   

title                        Waiting to Exhale  Father of the Bride Part II  \
title     

In [9]:
# [Study]
def recommendations(target_title, cos_sim_df=cos_sim_df, movie_data=movie_data, k=10):
    # print(cos_sim_df.loc[:, target_title].values.reshape(1, -1).argsort()[:, ::-1].flatten())
    recom_idx=cos_sim_df.loc[:, target_title].values.reshape(1, -1).argsort()[:, ::-1].flatten()[1:k+1]
    recom_title=movie_data.iloc[recom_idx, :].title.values
    recom_genre=movie_data.iloc[recom_idx, :].genres.values
    target_title_list=np.full(k, target_title)
    target_genre_list=np.full(k, movie_data[movie_data.title==target_title].genres.values)
    d={
        "target_title" : target_title_list,
        "target_genre" : target_genre_list,
        "recom_title" : recom_title,
        "recom_genre" : recom_genre
    }
    return pd.DataFrame(d)

In [10]:
recommendations("The Dark Knight Rises")

,target_title,target_genre,recom_title,recom_genre
0,The Dark Knight Rises,Action Crime Drama Thriller,The Dark Knight,Drama Action Crime Thriller
1,The Dark Knight Rises,Action Crime Drama Thriller,The Burglar,Crime Drama
2,The Dark Knight Rises,Action Crime Drama Thriller,Batman Begins,Action Crime Drama
3,The Dark Knight Rises,Action Crime Drama Thriller,Batman & Robin,Action Crime Fantasy
4,The Dark Knight Rises,Action Crime Drama Thriller,Batman,Fantasy Action
5,The Dark Knight Rises,Action Crime Drama Thriller,Raffles,Adventure Comedy Crime Drama History Romance T...
6,The Dark Knight Rises,Action Crime Drama Thriller,Hero at Large,Action Comedy Drama
7,The Dark Knight Rises,Action Crime Drama Thriller,DC Showcase: Catwoman,Action Adventure Animation Science Fiction
8,The Dark Knight Rises,Action Crime Drama Thriller,DC Super Hero Girls: Hero of the Year,Animation
9,The Dark Knight Rises,Action Crime Drama Thriller,Batman Returns,Action Fantasy
